# Project 3
## Data 620
### Jane Song

## Introduction 

For this assignment, I will be building a classifier to predict whether a name is male or female using the Names Corpus.

## Load the corpus and split the data
The Names Corpus contains names labeled as either male or female. This assignment specifies three subsets:
- Training set: 6,900 names
- Dev-test set: 500 names
- Test set: 500 names

Thus, the corpus will be split accordingly first. The training set is used to fit the classifier. The dev-test set helps compare feature approaches while developing the model. The test set is kept separate until the final evaluation.

In [8]:
import random
import nltk
from nltk.corpus import names

nltk.download('names')

# Combine the male and female names and attach their labels
labeled_names = (
    [(name, 'male') for name in names.words('male.txt')] +
    [(name, 'female') for name in names.words('female.txt')]
)

# Shuffle the names so the datasets contain a mixture of labels
random.seed(42)
random.shuffle(labeled_names)

print("Total names:", len(labeled_names))


Total names: 7944


[nltk_data] Downloading package names to
[nltk_data]     C:\Users\janej\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


In [9]:
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:7900]


## Initial Baseline Classifier 

The NLTK textbook notes that name one pattern of genderizing names is in the last letter or suffix of a name. Typically, names that end in vowels like a,e, and i are more likely to be female while names ending in consenants like k, o, r, s, and t or more likely to be male.  

The textbook begins with a simple feature function that uses only the last letter of a name. The feature function returns a dictionary because NLTK classifiers expect features in this format.

The training and dev-test names will be converted into feature sets, train a Naive Bayes classifier, and measure its accuracy on the dev-test set. This provides a baseline for comparison with later versions.


In [10]:
def gender_features(name):
    return {'last_letter': name[-1].lower()}

train_features = [(gender_features(name), gender) for name, gender in train_names]
devtest_features = [(gender_features(name), gender) for name, gender in devtest_names]

baseline_classifier = nltk.NaiveBayesClassifier.train(train_features)

baseline_accuracy = nltk.classify.accuracy(
    baseline_classifier, devtest_features
)

print("Baseline dev-test accuracy:", round(baseline_accuracy, 4))
print("Baseline dev-test percentage:", round(baseline_accuracy * 100, 2), "%")

Baseline dev-test accuracy: 0.754
Baseline dev-test percentage: 75.4 %


The textbook uses show_most_informative_features() to show which feature values provide the strongest evidence for one gender over the other. Since this baseline has only one feature (the last letter), the output shows which final letters are most strongly associated with each label.

In [4]:
baseline_classifier.show_most_informative_features(10)

Most Informative Features
             last_letter = 'a'            female : male   =     31.7 : 1.0
             last_letter = 'k'              male : female =     28.2 : 1.0
             last_letter = 'f'              male : female =     26.9 : 1.0
             last_letter = 'p'              male : female =     20.0 : 1.0
             last_letter = 'd'              male : female =      9.6 : 1.0
             last_letter = 'o'              male : female =      8.9 : 1.0
             last_letter = 'm'              male : female =      8.7 : 1.0
             last_letter = 'v'              male : female =      8.6 : 1.0
             last_letter = 'r'              male : female =      6.9 : 1.0
             last_letter = 'g'              male : female =      5.1 : 1.0


## Evaluate errors 
The textbook examines incorrect predictions to identify patterns the model is missing. Names in the dev-test set will be classified where the predicted gender does not match the actual label.

Reviewing these errors can suggest useful additional features, such as combinations of the last two letters rather than only the final letter.

In [14]:
baseline_errors = []

for name, actual_gender in devtest_names:
    predicted_gender = baseline_classifier.classify(gender_features(name))
    if predicted_gender != actual_gender:
        baseline_errors.append((actual_gender, predicted_gender, name))

print("Baseline errors:", len(baseline_errors))

for actual, predicted, name in sorted(baseline_errors)[:30]:
    print(f"actual={actual:<8} predicted={predicted:<8} name={name}")

Baseline errors: 123
actual=female   predicted=male     name=Beatriz
actual=female   predicted=male     name=Bliss
actual=female   predicted=male     name=Brigid
actual=female   predicted=male     name=Brooks
actual=female   predicted=male     name=Charil
actual=female   predicted=male     name=Cherlyn
actual=female   predicted=male     name=Cindelyn
actual=female   predicted=male     name=Coralyn
actual=female   predicted=male     name=Dallas
actual=female   predicted=male     name=Doralin
actual=female   predicted=male     name=Eden
actual=female   predicted=male     name=Eleanor
actual=female   predicted=male     name=Evangelin
actual=female   predicted=male     name=Gillan
actual=female   predicted=male     name=Ginnifer
actual=female   predicted=male     name=Glad
actual=female   predicted=male     name=Gretchen
actual=female   predicted=male     name=Grissel
actual=female   predicted=male     name=Harriott
actual=female   predicted=male     name=Hildagard
actual=female   predicte

## Expanded features 

To improve the baseline classifier, I will try to expand the feature extraction function to consider additional characteristics of each name. In addition to the last letter, I included the last two and three letters, the first letter, name length, and whether the name ends in a vowel.

These additional features can provide the classifier with more information about name patterns that may be associated with gender.

In [15]:
def gender_features2(name):
    name = name.lower()

    return {
        'last_letter': name[-1:],
        'last_two_letters': name[-2:],
        'last_three_letters': name[-3:],
        'first_letter': name[0],
        'length': len(name),
        'ends_in_vowel': name[-1] in 'aeiou'}

train_features2 = [(gender_features2(n), gender) for (n, gender) in train_names]
devtest_features2 = [(gender_features2(n), gender) for (n, gender) in devtest_names]

classifier2 = nltk.NaiveBayesClassifier.train(train_features2)

print("Improved dev-test accuracy:",
      nltk.classify.accuracy(classifier2, devtest_features2))

Improved dev-test accuracy: 0.794


## Additional suffix features
The textbook's next improvement uses the last one and two letters as suffix features. This is a focused test of the idea that short letter combinations may be more informative than a single final letter.

In [12]:
def suffix_features(name):
    name = name.lower()
    return {
        'suffix1': name[-1:],
        'suffix2': name[-2:] }

train_suffix_features = [(suffix_features(name), gender) for name, gender in train_names]
devtest_suffix_features = [(suffix_features(name), gender) for name, gender in devtest_names]

suffix_classifier = nltk.NaiveBayesClassifier.train(train_suffix_features)

suffix_accuracy = nltk.classify.accuracy(
    suffix_classifier, devtest_suffix_features)

print("Suffix-feature dev-test accuracy:", round(suffix_accuracy, 4))


Suffix-feature dev-test accuracy: 0.78


## Character count and character presence features
Another feature approach in the chapter looks at which letters occur in a name and how often they occur. The function below records the count of each letter and whether each letter appears at least once.

This gives the classifier information about the whole name, not just its beginning or ending. I will test this version on the dev-test set to see whether it adds useful information.

In [13]:
def character_features(name):
    name = name.lower()
    features = {
        'first_letter': name[0],
        'last_letter': name[-1],
        'suffix2': name[-2:]}

    for letter in 'abcdefghijklmnopqrstuvwxyz':
        features[f'count({letter})'] = name.count(letter)
        features[f'has({letter})'] = letter in name

    return features

train_character_features = [(character_features(name), gender) for name, gender in train_names]
devtest_character_features = [(character_features(name), gender) for name, gender in devtest_names]

character_classifier = nltk.NaiveBayesClassifier.train(train_character_features)

character_accuracy = nltk.classify.accuracy(
    character_classifier, devtest_character_features)

print("Character-feature dev-test accuracy:", round(character_accuracy, 4))

Character-feature dev-test accuracy: 0.792


## Compare the feature approaches and selecting a model
The model with the highest dev-test accuracy will be selected for the final test evaluation.

In [70]:
model_results = {
    'Last letter (baseline)': (baseline_accuracy, baseline_classifier, gender_features),
    'Expanded features': (expanded_accuracy, expanded_classifier, gender_features2),
    'Last one and two letters (textbook suffixes)': (suffix_accuracy, suffix_classifier, suffix_features),
    'Character counts and presence': (character_accuracy, character_classifier, character_features)}

for model_name, (accuracy, _, _) in model_results.items():
    print(f"{model_name}: {accuracy:.4f}")

best_model_name = max(model_results, key=lambda model: model_results[model][0])
best_accuracy, best_classifier, best_feature_function = model_results[best_model_name]

print("\nSelected model:", best_model_name)
print("Selected dev-test accuracy:", round(best_accuracy, 4))

Last letter (baseline): 0.7540
Expanded features: 0.7940
Last one and two letters (textbook suffixes): 0.7800
Character counts and presence: 0.7920

Selected model: Expanded features
Selected dev-test accuracy: 0.794


## Most informative features in the selected model

For the selected Naive Bayes model, I inspect the most informative features. These show which feature values are most strongly associated with one gender label compared with the other.

In [57]:
best_classifier.show_most_informative_features(10)

Most Informative Features
        last_two_letters = 'na'           female : male   =     93.7 : 1.0
        last_two_letters = 'la'           female : male   =     70.7 : 1.0
      last_three_letters = 'ard'            male : female =     43.4 : 1.0
        last_two_letters = 'rd'             male : female =     38.1 : 1.0
        last_two_letters = 'ia'           female : male   =     35.9 : 1.0
        last_two_letters = 'sa'           female : male   =     32.3 : 1.0
             last_letter = 'a'            female : male   =     31.7 : 1.0
        last_two_letters = 'ta'           female : male   =     29.3 : 1.0
      last_three_letters = 'nne'          female : male   =     29.1 : 1.0
             last_letter = 'k'              male : female =     28.2 : 1.0


## Error analysis of the selected model
Reviewing errors can help explain why some names are difficult to classify. Names may have uncommon spellings, endings shared by both labels, or characteristics that do not follow the patterns learned from the training data.

In [58]:
best_devtest_errors = []

for name, actual_gender in devtest_names:
    predicted_gender = best_classifier.classify(best_feature_function(name))
    if predicted_gender != actual_gender:
        best_devtest_errors.append((actual_gender, predicted_gender, name))

print("Selected model dev-test errors:", len(best_devtest_errors))

for actual, predicted, name in sorted(best_devtest_errors)[:30]:
    print(f"actual={actual:<8} predicted={predicted:<8} name={name}")

Selected model dev-test errors: 103
actual=female   predicted=male     name=Ainsley
actual=female   predicted=male     name=Beatriz
actual=female   predicted=male     name=Bliss
actual=female   predicted=male     name=Brigid
actual=female   predicted=male     name=Brooks
actual=female   predicted=male     name=Charil
actual=female   predicted=male     name=Cherey
actual=female   predicted=male     name=Cordy
actual=female   predicted=male     name=Dallas
actual=female   predicted=male     name=Debby
actual=female   predicted=male     name=Doralin
actual=female   predicted=male     name=Dory
actual=female   predicted=male     name=Eden
actual=female   predicted=male     name=Eleanor
actual=female   predicted=male     name=Elly
actual=female   predicted=male     name=Evangelin
actual=female   predicted=male     name=Gillan
actual=female   predicted=male     name=Ginnifer
actual=female   predicted=male     name=Glad
actual=female   predicted=male     name=Gretchen
actual=female   predicte

## Final evaluation on the test set
After selecting the expanded features model using dev-test accuracy, it will be used on the test set.

In [59]:
test_features = [
    (best_feature_function(name), gender)
    for name, gender in test_names]

test_accuracy = nltk.classify.accuracy(best_classifier, test_features)

test_errors = []

for name, actual_gender in test_names:
    predicted_gender = best_classifier.classify(best_feature_function(name))
    if predicted_gender != actual_gender:
        test_errors.append((actual_gender, predicted_gender, name))

print("Selected model:", best_model_name)
print("Dev-test accuracy:", round(best_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Number of test errors:", len(test_errors))
print("Number of correct test predictions:", len(test_names) - len(test_errors))

Selected model: Expanded features
Dev-test accuracy: 0.794
Test accuracy: 0.778
Number of test errors: 111
Number of correct test predictions: 389


## Conclusion 

The baseline classifier, which used only the last letter of each name, achieved 75.4% accuracy on the dev-test set. I then tested several approaches using additional features. The expanded-feature classifier performed best, achieving 79.4% accuracy, compared with 78.0% for the suffix-based approach and 79.2% for the character-feature approach.

I selected the expanded-feature classifier for the final evaluation. It achieved 77.8% accuracy on the test set, correctly classifying 389 out of 500 names, with 111 incorrect predictions.

The test accuracy was slightly lower than the dev-test accuracy (79.4%), a difference of 1.6 percentage points. This difference may be due to variation in the names included in each dataset, since some names may have patterns that are more difficult for the classifier to identify. The dev-test accuracy may also be slightly optimistic because it was used repeatedly to compare and select features. Overall, the small difference suggests that the classifier performed fairly consistently on both datasets.